> 本 Notebook 由对应 Word 实验手册生成。只有“测试与验收”章节中的测试指令可执行；其余代码仅用于阅读和讲解。


# 实验五：基于CANN的GQA-Attention基础版算子实验


建议学时：4学时


# 实验任务


## 任务描述


本实验围绕Qwen2.5的Grouped Query Attention (GQA)计算构建AscendC自定义算子。算子数学语义为：，其中Q有个head、K/V各有个head。实验主要指导从kernel/host端Ascend代码开发、直调核函数在PyTorch中的注册、单算子数学正确性验证，最终嵌入Qwen2.5-0.5B模型中通过实机计时形成完整开发流程闭环。


算子在causal模式下支持Qwen的右上三角掩码语义，query位置p只能访问k≤p的key，decode阶段（=1, >1）使用右对齐窗口。基础版使用在线softmax，通过标量GM访问逐元素计算QK点积和V加权累积。


## 学习目标


完成本任务的学习后，你应当能理解GQA Attention的数学语义与Q/K/V张量布局、掌握在线softmax的数值稳定实现、基于Ascend C完成GQA Attention算子的设备端标量计算、完成PyTorch框架下直调核函数的注册与调用，处理GQA的批次、头、序列多维输入、设计覆盖causal/non-causal/decode三种场景的正确性测试方法、掌握基础Attention算子的完整开发流程。


# 任务准备


## 算子定义


Attention算子主要接受已进行线性变换、旋转编码所得的Q、K、V三者矩阵，进行经典的计算：


## GQA Attention的作用与计算过程


GQA属于注意力机制的变体结构，旨在缓解标准多头注意力在多轮推理时因Key-Value缓存过大而带来的显存压力和访存瓶颈。在标准的MHA中，每个查询头都独立对应一个KV头，因此每生成一个Token都需要缓存所有头的K、V矩阵，随着序列长度和头数的增长，KV缓存的占用会线性膨胀，成为长文本生成的主要资源瓶颈。GQA通过将多个Query头划分为若干组，每组内部共享同一个KV头，从而将KV缓存的总头数从Query头数缩减为分组数，大幅降低了显存占用和I/O开销。


以8个Query头为例，若采用4组分组（每组2个Query共享1个KV），则KV头数从8降为4，缓存量减半；若极端情况下采用单组（即MQA，多查询注意力），则KV头数降为1，缓存压缩最为显著。但MQA会因过度共享而损失模型表达能力，影响精度；GQA则通过可控的分组数在压缩效率和表示能力之间取得平衡，既保留了MHA的多头多样性，又大幅减少了推理时的内存搬运。实际实现中，GQA的K、V投影矩阵维度按照分组数设计，而Q保持原有头数，计算注意力时先将同一组内的Q与共享的K、V批量进行点积和加权求和，最终输出的每个头仍保持独立的语义子空间。这一设计使得GQA在长序列推理场景中能以微小的精度代价换取显著的吞吐量提升，目前已在大规模语言模型的推理加速中得到广泛应用。


## 算子定义与接口约定


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">当前工程约定</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">数据类型</td>
<td style="text-align:left;">Q、K、V、output 均为 float32</td>
</tr>
<tr>
<td style="text-align:left;">输入布局</td>
<td style="text-align:left;">四维连续 Tensor：Q [B, Hq, Sq, D]、K [B, Hkv, Sk, D]、V [B, Hkv, Sk, D]；output 同 Q 形 [B, Hq, Sq, D]；Hq 必须被 Hkv 整除；headDim 必须被 8 整除</td>
</tr>
<tr>
<td style="text-align:left;">行数</td>
<td style="text-align:left;">totalQueries = B × Hq × Sq</td>
</tr>
<tr>
<td style="text-align:left;">并行划分</td>
<td style="text-align:left;">coreNum = min(8, totalQueries)；queriesPerCore = (totalQueries + coreNum - 1) / coreNum 向上取整；每个 Core 从 coreId 推导起始的 (batch, qHead)，末核自动截断</td>
</tr>
<tr>
<td style="text-align:left;">标准测试形状</td>
<td style="text-align:left;">B=1, Hq=8, Hkv=2, Sq=Sk=32, D=64, causal=1, coreNum=8；scale 默认 1/√D</td>
</tr>
</tbody></table>


## 实验环境准备


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">项目</th>
<th style="text-align:left;">配置</th>
</tr>
</thead>
<tbody><tr>
<td style="text-align:left;">硬件</td>
<td style="text-align:left;">Ascend 910B4宿主NPU</td>
</tr>
<tr>
<td style="text-align:left;">工具链</td>
<td style="text-align:left;">CANN 8.5.0 需先source set_env.sh</td>
</tr>
<tr>
<td style="text-align:left;">框架接口</td>
<td style="text-align:left;">PyTorch C++ extension，torch.ops.gqa_attention_custom.gqa_attention(Tensor q, Tensor k, Tensor v, float scale=0., bool causal=True) -&gt; Tensor</td>
</tr>
<tr>
<td style="text-align:left;">构建结果</td>
<td style="text-align:left;">out/lib/libascendc_kernels_npu.so、libgqa_attention_torch_register.so；out/bin/gqa_attention_baseline_standalone</td>
</tr>
<tr>
<td style="text-align:left;">测试参考</td>
<td style="text-align:left;">PyTorch 标准Attention</td>
</tr>
</tbody></table>


# 任务实施


## 步骤一：定义I/O规格


GQA Attention的输入为Q、K、V三个四维张量，形状分别为[B, Hq, Sq, D]、[B, Hkv, Sk, D]、[B, Hkv, Sk, D]，输出为[B, Hq, Sq, D]且不改变形状。必须能被整除（Qwen2.5-0.5B中=14, =2，比例为7:1）。scale因子默认为。


Tiling结构体需传递以下运行时参数：batch、queryHeads、kvHeads、queryLen、keyLen、headDim、totalQueries、coreNum、queriesPerCore、causal和scale。scale默认为0（Host端自动替换为），Host端根据输入shape动态填充Tiling，kernel入口按uint32_t和float两种视角分别解包整数字段和scale浮点字段：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">#pragma pack(push, 1) struct GqaAttentionBaselineTiling { uint32_t batch = 0; uint32_t queryHeads = 0; uint32_t kvHeads = 0; uint32_t queryLen = 0; uint32_t keyLen = 0; uint32_t headDim = 0; uint32_t totalQueries = 0; uint32_t coreNum = 1; uint32_t queriesPerCore = 1; uint32_t causal = 1; float scale = 1.0f; };</th>
</tr>
</thead>
</table>


每个Core负责若干query head的计算。由于query之间无依赖，基础版按(batch, qHead) 对均分给各Core：coreId决定起始的batch和qHead索引，rowsPerCore决定每个Core处理的query数量，末核自动截断尾部。


## 步骤二：编写AscendC Kernel


基础版kernel对每个(batch, qHead, qPos)三元组执行一次完整的attention计算。首先确定对应的kvHead，然后计算causal下的validKeys。在validKeys范围内逐key位置执行标量QK点积、乘以scale、通过自包含的GqaBaselineExp实现在线softmax更新，最终除以normalizer归一化。核心循环摘录自op_kernel：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">const uint32_t kvHead = qHead / (queryHeads_ / kvHeads_); uint32_t validKeys = keyLen_; if (causal_ != 0) { int32_t visible = qPos + keyLen_ - queryLen_ + 1; validKeys = visible &lt;= 0 ? 0 : min((uint32_t)visible, keyLen_); } float maxScore = -3.4e38f, normalizer = 0.0f; for (uint32_t ki = 0; ki &lt; validKeys; ++ki) { // QK dot product float score = 0.0f; for (uint32_t d = 0; d &lt; headDim_; ++d) score += queryGm_.GetValue(qBase+d) * keyGm_.GetValue(kBase+d); score *= scale_; // online softmax: oldFactor/newFactor update float nextMax = (score &gt; maxScore) ? score : maxScore; float oldFactor = GqaBaselineExp(maxScore - nextMax); float newFactor = GqaBaselineExp(score - nextMax); normalizer = normalizer * oldFactor + newFactor; // V accumulation: outputGm_ in-place update for (uint32_t d = 0; d &lt; headDim_; ++d)</th>
</tr>
</thead>
</table>


在线softmax使用数值稳定的两变量追踪（maxScore与normalizer），output在原地乘oldFactor后累加newFactor * V的方式更新，避免物化 [Sq, Sk] 的完整score矩阵。每处理一个key后仅做两次GqaBaselineExp和一次乘加更新，最终除以normalizer归一化。该路径刻意保留直接GM标量访问，每query对每key执行headDim次GetValue读取Q和K、softmax更新后对headDim次GetValue读取V并SetValue原地更新output。


## 步骤三：编写Host端调用与算子注册


Host端分为standalone验证程序和torch_extension注册文件。


standalone完成完整ACL生命周期，torch_extension通过TORCH_LIBRARY注册为torch.ops.gqa_attention_custom.gqa_attention。注册前进行防御性校验：


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">TORCH_CHECK(q.device().is_cpu() &amp;&amp; k.device().is_cpu() &amp;&amp; v.device().is_cpu()); TORCH_CHECK(q.dim() == 4 &amp;&amp; k.dim() == 4 &amp;&amp; v.dim() == 4); TORCH_CHECK(q.size(1) % k.size(1) == 0, &quot;Hq must be divisible by Hkv&quot;); TORCH_CHECK(q.size(3) % 8 == 0, &quot;headDim must be divisible by 8&quot;); TORCH_LIBRARY(gqa_attention_custom, m) { m.def(&quot;gqa_attention(Tensor q, Tensor k, Tensor v,&quot; &quot; float scale=0., bool causal=True) -&gt; Tensor&quot;); } TORCH_LIBRARY_IMPL(gqa_attention_custom, CompositeExplicitAutograd, m) { m.impl(&quot;gqa_attention&quot;, gqa_attention_baseline_npu); }</th>
</tr>
</thead>
</table>


## 步骤四：CMake目标构建


构建系统通过CMake ExternalProject机制先由ascendc_library经bisheng编译Device端kernel为libascendc_kernels_npu.so，再用g++分别编译standalone可执行文件和torch注册动态库。注意：基础版Attention的torch wrapper采用每次调用时动态aclrtMalloc/aclrtFree的方式（与RoPE的静态全局缓存不同），构建时需将aclrtlaunch_*.h和libascendc_kernels_npu.so的路径加入include/lib搜索链。


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">aclrtRecordEvent(start, stream); for (uint32_t i = 0; i &lt; repeat; ++i) ACLRT_LAUNCH_KERNEL(gqa_attention_baseline_kernel)(blockDim, stream, ...); aclrtRecordEvent(stop, stream); aclrtEventElapsedTime(&amp;elapsedMs, start, stop); const double us = elapsedMs * 1000.0 / repeat;</th>
</tr>
</thead>
</table>


<table style="margin-left:0 !important; margin-right:auto !important; text-align:left !important;">
<thead>
<tr>
<th style="text-align:left;">cd /YOURPATH/GqaAttentionBaselineExperiment bash scripts/check_env.sh bash scripts/build.sh bash scripts/run_test.sh tests/test_torch_op.py # 单算子正确性 bash scripts/run_test.sh tests/test_qwen_forward.py # Qwen链路替换验证 bash scripts/compare_qwen2_5_forward.sh # Qwen2.5前向vs native对比 BATCH=1 Q_HEADS=8 KV_HEADS=2 Q_LEN=32 KV_LEN=32 HEAD_DIM=64 BLOCK_DIM=8 WARMUP=10 REPEAT=50 ROUNDS=5 CAUSAL=1 bash scripts/profile.sh</th>
</tr>
</thead>
</table>


# 测试与验收


## 一、单算子正确性测试


该测试调用当前实验的PyTorch注册算子，并与同一数学语义的参考实现比较。终端输出全部为PASS（或ALL PASS）且进程返回码为0，表示正确性测试通过。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GqaAttentionBaselineExperiment
bash scripts/build.sh
python3 tests/test_torch_op.py


## 二、单算子执行时间测试


该指令先预热，再重复启动单个算子，并使用ACL Event统计设备侧执行时间。记录输出中的mean、median、min和max；该结果不包含Python参考计算、输入生成、结果比对及首次主机到设备的数据传输。基础版与优化版比较时，应使用相同输入形状、预热次数、重复次数和计算核心数。


In [ ]:
%%bash
set -e

cd /mnt/workspace/cann-learning-hub/contrib/tutorials/qwen_ops
source ./setup_cannlab_env.sh
cd Qwen2.5cann_ops/GqaAttentionBaselineExperiment
export LD_LIBRARY_PATH="$PWD/out/lib:${LD_LIBRARY_PATH:-}"
./out/bin/gqa_attention_baseline_standalone --batch 1 --q-heads 8 --kv-heads 2 --q-len 32 --kv-len 32 --head-dim 64 --block-dim 8 --causal 1 --warmup 10 --repeat 50 --rounds 5


# 任务拓展


改变batch、序列长度、head数和head_dim采样，比较不同配置下的device时间；使用msprof分析kernel调度、访存与AI Core利用率。优化版可继续探索K-tile UB复用、向量化QK点积、Cube/MatMul加速或更精细的softmax tiling。


# 实验总结


本实验完成了GQA Attention基础版算子的完整开发闭环：从GQA的Hq/Hkv映射关系和causal mask语义出发，定义Q/K/V四维张量规格与Tiling数据；编写AscendC kernel实现逐query对标量QK点积、在线softmax和V加权累积；构建Host端独立可执行验证程序和torch.library注册动态库；通过PyTorch causal attention参考实现验证causal/non-causal/decode三种场景全部在3e-3阈值内通过；最后使用ACL Event计时获得684.1 us（B=1,Hq=8,Hkv=2,Sq=Sk=32,D=64）的device侧kernel性能基线。


基础版的核心价值在于其功能基本实现，每个query对每个key的headDim次Q读 + headDim次K读+softmax更新+headDim次V读+headDim次out写，完整对应注意力计算公式的逐元素展开，无UB缓冲复用、无K-tile共享、无向量化或Cube加速。通过阅读kernel代码即可逐行对账在线softmax的三变量追踪（m/l/acc）机制。


本实验建立了Attention算子开发的标准流程：


定义GQA规格 → 编写kernel（在线softmax）→ 注册调用 → 构建验证 → Golden测试（三场景覆盖）→ 计时分析，适用于后续Attention优化版（K-tile复用、Cube加速）的开发任务。
